# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sumit07-git/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Rule

I will prioritize content that appears stale but still has meaningful search visibility.

The rule uses two signals:

1. `days_since_update` — measures how long it has been since the content was updated.
2. `impressions` — measures the amount of search visibility the content receives.

The idea is that older content with continued search visibility may represent a useful refresh opportunity. This is a simple decision-support baseline, not a claim that every stale page needs updating.

### Reason code

- `stale_but_visible` — the content is at least 180 days old and has at least 500 impressions, so it is prioritized for refresh.
- `not_prioritized` — the content does not meet both conditions.

### Action labels

- `REFRESH` — prioritize the content for a refresh review.
- `NO_ACTION` — do not prioritize it under this baseline rule.

The thresholds are deliberately hand-written and transparent rather than learned from the outcome label.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

required_cols = ["days_since_last_update", "impressions_90d"]

missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    print("Missing columns:", missing_cols)
else:
    print("All required columns are available.")




Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
All required columns are available.


### Baseline scoring rule

A content item receives a positive score only when both conditions are satisfied:

- `days_since_update >= 180`
- `impressions >= 500`

The score is based on impressions for qualifying stale content. This gives higher priority to stale content that still has greater search visibility.

The rule produces one reason code and one action label for every row, then ranks the complete dataset from highest to lowest score.

The rule does not use future-window metrics, `is_declining_label`, `trend_direction`, `trend_pct`, or product/client flags.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [25]:
queue = df.copy()
STALE_DAYS = 180
MIN_IMPRESSIONS = 500
queue["days_since_last_update"] = pd.to_numeric(
    queue["days_since_last_update"],
    errors="coerce"
)

queue["impressions_90d"] = pd.to_numeric(
    queue["impressions_90d"],
    errors="coerce"
)

queue["is_stale"] = (
    queue["days_since_last_update"] >= STALE_DAYS
)

queue["is_visible"] = (
    queue["impressions_90d"] >= MIN_IMPRESSIONS
)

queue["score"] = np.where(
    queue["is_stale"] & queue["is_visible"],
    queue["impressions_90d"],
    0
)

queue["reason_code"] = np.where(
    queue["score"] > 0,
    "stale_but_visible",
    "not_prioritized"
)

queue["action"] = np.where(
    queue["score"] > 0,
    "REFRESH",
    "NO_ACTION"
)

queue = queue.sort_values(
    ["score", "impressions_90d", "days_since_last_update"],
    ascending=[False, False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

print("Rows ranked:", len(queue))

display(
    queue[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d"
        ]
    ].head(10)
)

output_cols = [
    "content_id",
    "score",
    "reason_code",
    "action",
    "rank"
]

output_path = "work/outputs/baseline_action_score.csv"

queue[output_cols].to_csv(
    output_path,
    index=False
)

print("CSV successfully written:")
print(output_path)

print("Rows written:", len(queue))

Rows ranked: 30000


,rank,content_id,score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,61678,stale_but_visible,REFRESH,194,61678
1,2,content_7368877ea310,59472,stale_but_visible,REFRESH,194,59472
2,3,content_1bfaa38ff26c,25715,stale_but_visible,REFRESH,194,25715
3,4,content_0a91db491d14,13299,stale_but_visible,REFRESH,193,13299
4,5,content_5feee3994adb,7812,stale_but_visible,REFRESH,194,7812
5,6,content_c2d929d83eaa,7558,stale_but_visible,REFRESH,193,7558
6,7,content_b16bd7307b39,4590,stale_but_visible,REFRESH,194,4590
7,8,content_fe16a55cd13d,4556,stale_but_visible,REFRESH,194,4556
8,9,content_ecb6215e79fd,4429,stale_but_visible,REFRESH,194,4429
9,10,content_928af3e22c80,1697,stale_but_visible,REFRESH,193,1697


CSV successfully written:
work/outputs/baseline_action_score.csv
Rows written: 30000


## 3. Top-20 review

I reviewed the top 20 ranked items using the baseline's action, reason code, staleness, and search visibility.

The baseline recommends `REFRESH` when an item is at least 180 days since its last update and has at least 500 impressions over the last 90 days.

For each item, I record:
- the action suggested by the rule,
- the reason code,
- why the item was ranked,
- a confidence note based on the observed signals,
- and what could make the recommendation wrong.

These are decision-support recommendations rather than claims that the content definitely needs updating.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = queue.head(20).copy()

review_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d"
]

display(top20[review_columns])


,rank,content_id,score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,61678,stale_but_visible,REFRESH,194,61678
1,2,content_7368877ea310,59472,stale_but_visible,REFRESH,194,59472
2,3,content_1bfaa38ff26c,25715,stale_but_visible,REFRESH,194,25715
3,4,content_0a91db491d14,13299,stale_but_visible,REFRESH,193,13299
4,5,content_5feee3994adb,7812,stale_but_visible,REFRESH,194,7812
5,6,content_c2d929d83eaa,7558,stale_but_visible,REFRESH,193,7558
6,7,content_b16bd7307b39,4590,stale_but_visible,REFRESH,194,4590
7,8,content_fe16a55cd13d,4556,stale_but_visible,REFRESH,194,4556
8,9,content_ecb6215e79fd,4429,stale_but_visible,REFRESH,194,4429
9,10,content_928af3e22c80,1697,stale_but_visible,REFRESH,193,1697


## 4. Weak picks + leakage check

### Weak picks

The baseline can produce weak recommendations because it uses only content staleness and search visibility.

A high score does not prove that a page needs to be updated. A page may be intentionally evergreen, already accurate despite being old, or receive impressions because of temporary search demand.

I therefore treat the ranking as decision-support rather than an automatic refresh decision.

### Leakage check

The baseline score uses only:

- `days_since_last_update`
- `impressions_90d`

The score does not use future-window metrics, `trend_direction`, `trend_pct`, outcome labels, or product/client flags.

The ranking is therefore based on the two hand-written signals rather than information derived from the future outcome.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

weak_picks = top20[
    (top20["days_since_last_update"] < STALE_DAYS + 60) &
    (top20["impressions_90d"] < MIN_IMPRESSIONS * 2)
].copy()

print("Potential weak picks from the Top-20:")

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d"
        ]
    ]
)


Potential weak picks from the Top-20:


,rank,content_id,score,reason_code,action,days_since_last_update,impressions_90d
13,14,content_77d4d5930e5e,828,stale_but_visible,REFRESH,194,828
15,16,content_6226ee6adc91,545,stale_but_visible,REFRESH,183,545
16,17,content_074ba6ead17b,533,stale_but_visible,REFRESH,183,533


In [28]:
score_inputs = [
    "days_since_last_update",
    "impressions_90d"
]

print("Signals used in baseline score:")
for col in score_inputs:
    print(" -", col)

forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("\nPotential label/future-related columns:")
for col in forbidden_columns:
    if col in queue.columns:
        print(f" - {col}: PRESENT in dataset, NOT USED in score")
    else:
        print(f" - {col}: NOT PRESENT")

print("\nLeakage check complete.")

Signals used in baseline score:
 - days_since_last_update
 - impressions_90d

Potential label/future-related columns:
 - trend_direction: PRESENT in dataset, NOT USED in score
 - trend_pct: PRESENT in dataset, NOT USED in score
 - is_declining_label: NOT PRESENT

Leakage check complete.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.